In [1]:
from haystack import Pipeline,Document
from haystack.components.routers import ConditionalRouter
from haystack.components.websearch.serper_dev import SerperDevWebSearch
from haystack.components.builders import ChatPromptBuilder
from haystack.dataclasses import ChatMessage
from haystack.components.generators.chat import OpenAIChatGenerator

In [13]:
documents = [
    Document(
        content="""Munich, the vibrant capital of Bavaria in southern Germany, exudes a perfect blend of rich cultural
                                heritage and modern urban sophistication. Nestled along the banks of the Isar River, Munich is renowned
                                for its splendid architecture, including the iconic Neues Rathaus (New Town Hall) at Marienplatz and
                                the grandeur of Nymphenburg Palace. The city is a haven for art enthusiasts, with world-class museums like the
                                Alte Pinakothek housing masterpieces by renowned artists. Munich is also famous for its lively beer gardens, where
                                locals and tourists gather to enjoy the city's famed beers and traditional Bavarian cuisine. The city's annual
                                Oktoberfest celebration, the world's largest beer festival, attracts millions of visitors from around the globe.
                                Beyond its cultural and culinary delights, Munich offers picturesque parks like the English Garden, providing a
                                serene escape within the heart of the bustling metropolis. Visitors are charmed by Munich's warm hospitality,
                                making it a must-visit destination for travelers seeking a taste of both old-world charm and contemporary allure."""
    )
]

In [2]:
import os
from dotenv import load_dotenv
load_dotenv(override = True)
api_key = os.getenv('OPENAI_API_KEY')
serper_key=os.getenv("SERPERDEV_API_KEY")

In [3]:
template = [
    ChatMessage.from_user(
        """
        Answer the following question based on the documents given,if answer not found from the document reply 'no_answer'

        {% for doc in documents %}
        {{ doc.content }}
        {% endfor %}

        question : {{question}}
        answer :
        """
    )
]
builder = ChatPromptBuilder(template = template)

In [4]:
llm = OpenAIChatGenerator(model = "gpt-4o-mini")

In [5]:
routes = [
    {
    "condition" : "{{ 'no_answer' in replies[0].text}}",
    "output"    : "{{query}}",
    "output_name": "go_to_websearch",
    "output_type": str,
    },
    {
    "condition" : "{{ 'no_answer' not in replies[0].text}}",
    "output"    : "{{replies[0].text}}",
    "output_name": "answer",
    "output_type": str,
    }
]
router = ConditionalRouter(routes=routes)

In [6]:
websearch = SerperDevWebSearch()

In [7]:
web_template = [
    ChatMessage.from_user(
        """
        Answer the following question based on the documents given

        {% for doc in documents %}
        {{ doc.content }}
        {% endfor %}

        question : {{question}}
        answer :
        """
    )
]
web_builder = ChatPromptBuilder(template = web_template)

In [8]:
websearch_llm = OpenAIChatGenerator(model = "gpt-4o-mini")

In [9]:
pipeline = Pipeline()

pipeline.add_component("builder",builder)
pipeline.add_component("llm",llm)
pipeline.add_component("router",router)
pipeline.add_component("websearch",websearch)
pipeline.add_component("web_builder",web_builder)
pipeline.add_component("websearch_llm",websearch_llm)



In [12]:
pipeline.connect("builder.prompt","llm.messages")
pipeline.connect("llm.replies","router.replies")
pipeline.connect("router.go_to_websearch","websearch.query")
pipeline.connect("router.go_to_websearch","web_builder.question")
pipeline.connect("websearch.documents","web_builder.documents")
pipeline.connect("web_builder.prompt","websearch_llm.messages")

🚅 Components
  - builder: ChatPromptBuilder
  - llm: OpenAIChatGenerator
  - router: ConditionalRouter
  - websearch: SerperDevWebSearch
  - web_builder: ChatPromptBuilder
  - websearch_llm: OpenAIChatGenerator
🛤️ Connections
  - builder.prompt -> llm.messages (List[ChatMessage])
  - llm.replies -> router.replies (List[ChatMessage])
  - router.go_to_websearch -> websearch.query (str)
  - router.go_to_websearch -> web_builder.question (str)
  - websearch.documents -> web_builder.documents (List[Document])
  - web_builder.prompt -> websearch_llm.messages (List[ChatMessage])

In [19]:
query_from_doc = "Where is Munich?"
query_notfrom_doc = "How many people live in Munich?"

result = pipeline.run({"builder" : {"question":query,"documents":documents},"router":{"query":query}})
print(result['router']['answer'])

Munich is located in southern Germany, along the banks of the Isar River.


In [21]:
result = pipeline.run({"builder" : {"question":query_notfrom_doc,"documents":documents},"router":{"query":query_notfrom_doc}})

{'websearch': {'links': ['https://en.wikipedia.org/wiki/Munich',
   'https://www.britannica.com/place/Munich-Bavaria-Germany',
   'https://en.wikipedia.org/wiki/Demographics_of_Munich',
   'https://www.macrotrends.net/global-metrics/cities/204371/munich/population',
   'https://www.statista.com/statistics/505774/munich-population/',
   'https://eurocities.eu/cities/munich/',
   'https://www.coe.int/en/web/interculturalcities/munich',
   'https://stadt.muenchen.de/dam/jcr:459cce24-3894-4d4d-b144-4185b750a310/LHM.Stat_Faltkarte_2019_englisch.pdf',
   'https://www.ricksteves.com/watch-read-listen/read/articles/munich-a-metropolis-with-smalltown-charm']},
 'websearch_llm': {'replies': [ChatMessage(_role=<ChatRole.ASSISTANT: 'assistant'>, _content=[TextContent(text='As of 30 November 2024, the population of Munich is 1,604,384 inhabitants.')], _name=None, _meta={'model': 'gpt-4o-mini-2024-07-18', 'index': 0, 'finish_reason': 'stop', 'usage': {'completion_tokens': 23, 'prompt_tokens': 395, '

In [22]:
print(result['websearch_llm']['replies'][0].text)

As of 2023, Munich has a population of roughly 1.51 million people. As of 30 November 2024, the population is reported to be 1,604,384 inhabitants.
